1) Setup + Authentication

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,StructField,
    StringType,IntegerType,DoubleType,TimestampType
) 
from pyspark.sql.window import Window

In [0]:
storage_account_name = 'stretailanalytics'

bronze_staged=f'abfss://bronze@{storage_account_name}.dfs.core.windows.net/staged/'
bronze_customer=f'abfss://bronze@{storage_account_name}.dfs.core.windows.net/staged/customers/'
bronze_items=f'abfss://bronze@{storage_account_name}.dfs.core.windows.net/staged/order_items/'
bronze_orders=f'abfss://bronze@{storage_account_name}.dfs.core.windows.net/staged/orders/'
bronze_payments=f'abfss://bronze@{storage_account_name}.dfs.core.windows.net/staged/payments/'

In [0]:

client_id = '25c```````````````````'
client_secret='rfK8Q````````````````````'
tenant_id='e603`````````````'

In [0]:
spark.conf.set(f'fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net','OAuth')
spark.conf.set(f'fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net','org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider')
spark.conf.set(f'fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net',client_id)
spark.conf.set(f'fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net',client_secret)
spark.conf.set(f'fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net',f'https://login.microsoftonline.com/{tenant_id}/oauth2/token')

print("Configured successfully")

2) Read Order CSV

In [0]:
order_schema=StructType([
    StructField("order_id",     StringType(),nullable=False),
    StructField("customer_id",  StringType(),nullable=True),
    StructField("order_status", StringType(),nullable=True),
    StructField("order_purchase_timestamp", TimestampType(),nullable=True),
    StructField("order_approved_at",TimestampType(),nullable=True),
    StructField("order_delivered_carrier_date",TimestampType(),nullable=True),
    StructField("order_delivered_customer_date",TimestampType(),nullable=True),
    StructField("order_estimated_delivery_date",TimestampType(),nullable=True)

])


In [0]:
df_raw_orders=spark.read\
    .schema(order_schema)\
        .option("header",'True')\
            .option('timestampFormat','yyyy-MM-dd HH:mm:ss')\
                .option('mode','PERMISSIVE')\
                    .csv(bronze_orders)

In [0]:
display(df_raw_orders.limit(10))

In [0]:
df_raw_orders.count()

3) Data Quality check

In [0]:
#Null count per column using list comprehension
null_count=df_raw_orders.select([
    F.count(F.when(F.col(c).isNull(),c)).alias(c)  #F.when(condition, value)
    for c in df_raw_orders.columns
])

display(null_count)

In [0]:
##Distinct Order_status value
Order_status= df_raw_orders.groupBy('order_status')\
    .agg(F.count('*').alias("Each_cat_count"))\
        .orderBy("Each_cat_count",ascending=False)

display(Order_status)

In [0]:
##Date Range
df_raw_orders.select(
    F.min('order_purchase_timestamp').alias('earliest_order'),
    F.max('order_purchase_timestamp').alias('latest_order')
).show(truncate=False)

Apply Transformation


In [0]:
(df_raw_orders.columns)

In [0]:
df_silver_orders=df_raw_orders\
    .dropna(subset=['order_id'])\
    .dropDuplicates(['order_id'])\
    .withColumn('order_year', F.year('order_purchase_timestamp'))\
    .withColumn('order_month',F.month('order_purchase_timestamp'))\
    .withColumn('order_day',F.day('order_purchase_timestamp'))\
    .withColumn('order_quarter',F.quarter('order_purchase_timestamp'))\
    .withColumn('order_hour',F.hour('order_purchase_timestamp'))\
    .withColumn('order_status',F.trim(F.lower(F.col('order_status'))))\
    .withColumn('delivery_status',F.when(F.col('order_status')=='delivered', 'Completed')
                                    .when(F.col('order_status')=='shipped', 'In Transist')
                                    .when(F.col('order_status')=='canceled','Cancelled')
                                    .when(F.col('order_status')=='invoiced', 'Invoiced')
                                    .when(F.col('order_status')=='processing','processing')
                                    .otherwise('Other')
                                                )\
    .withColumn('days_to_deliver',F.when(F.col('order_delivered_customer_date').isNotNull(), 
                                        F.datediff(F.col('order_delivered_customer_date'),
                                                    F.col('order_purchase_timestamp')
                                                    )).otherwise(None))\
    .withColumn('is_late_delivery',F.when(F.col('order_delivered_customer_date')>F.col('order_estimated_delivery_date'),True).otherwise(False))\
    .withColumn('ingestion_Timestamp',F.current_timestamp())\
    .withColumn('ingestion_date',F.current_date())\
    .withColumn('source_system', F.lit('olist_ecommerce'))\
    .withColumn('pipeline_name',F.lit('bronze_to_silver'))\
    .select(
        'order_id',
        'customer_id',
        'order_status',
        'delivery_status',
        'order_purchase_timestamp',
        'order_approved_at',
        'order_delivered_customer_date',
        'order_estimated_delivery_date',
        'order_year',
        'order_month',
        'order_day',
        'order_quarter',
        'order_hour',
        'days_to_deliver',
        'is_late_delivery',
        'ingestion_timestamp',
        'ingestion_date',
        'source_system',
        'pipeline_name'
    )

print(f'Silver orders row count :{df_silver_orders.count()}')  
display(df_silver_orders.limit(10))                                                                              

Quarantine bad rows

In [0]:
valid_statuses=['delivered','shipped','canceled','unavailable','invoiced','processing','created','approved']

df_quarantine=df_raw_orders.filter(
    ~F.col('order_status').isin(valid_statuses) | F.col('order_id').isNull()
)

In [0]:
display(df_quarantine)

In [0]:
quarantine_count=df_quarantine.count()
print(f'Quarantine row count : {quarantine_count}')

In [0]:
if quarantine_count > 0:
    df_quarantine\
    .withColumn('Quarantine_reason',F.lit('invalid_statur_or_null_pk'))\
    .withColumn('Quarantine_timestamp',F.current_timestamp())\
    .write\
        .format('delta')\
        .mode('append')\
        .save(f'abfss://silver@{storage_account_name}.dfs.core.windows.net/quarantine/orders/')
    print(f'{quarantine_count} rows written to quarantine')
else:
    print(f'No quarantine rows-all data valid')

Write silver Delta table

In [0]:
#Write to Silver as Delta (partitioned by year + month)
df_silver_orders.write\
    .format('delta')\
    .mode('overwrite')\
    .option('overwriteSchema','true')\
    .partitionBy('order_year','order_month')\
    .save(f'abfss://silver@{storage_account_name}.dfs.core.windows.net/SILVER_ORDERS/')

print('Silver order delta table written!')


Delta format = Parquet files + Transaction Log

Delta Table
    │
    ├── Parquet Data Files
    │
    └── _delta_log
            │
            ├── transaction logs
            ├── metadata
            ├── schema history
            └── version history

Delta_logs contains:

commits
operations
schema
table history

This is the “brain” of Delta Lake.

Most Important Enterprise Usage

Delta is heavily used in:

Medallion Architecture
Bronze/Silver/Gold layers
CDC pipelines
Streaming ETL
Lakehouse architecture

In [0]:
silver_path=f'abfss://silver@{storage_account_name}.dfs.core.windows.net/SILVER_ORDERS/'
display(dbutils.fs.ls(silver_path))

Register as SQL table and verify

In [0]:
#Register in metastore + verify with SQL query
spark.sql('create database if not exists retail_silver')

In [0]:
display(spark.sql('SHOW EXTERNAL LOCATIONS'))

In [0]:
catalog = spark.sql(
    "SELECT current_catalog()").collect()[0][0]
print(f"Catalog: {catalog}")

In [0]:
df_silver_check=spark.read\
    .format('delta')\
        .load(silver_path)

In [0]:
display(df_silver_check.limit(10))

Customers Silver

In [0]:
customer_schema=StructType([
    StructField('customer_id', StringType(),nullable=False),
    StructField('customer_unique_id', StringType(),nullable=True),
    StructField('customer_zip_code_prefix', IntegerType(), nullable=True),
    StructField('customer_city', StringType(), nullable=True),
    StructField('customer_state',StringType(),nullable=True)
])

In [0]:
df_raw_customer=spark.read\
    .schema(customer_schema)\
    .option('header',True)\
    .csv(bronze_customer)

In [0]:
display(df_raw_customer.limit(10))

In [0]:
df_silver_customer=df_raw_customer.dropna(subset=['customer_id'])\
    .dropDuplicates(['customer_id'])\
    .withColumn('customer_city',
                F.initcap(F.trim(F.col('customer_city'))))\
    .withColumn('customer_state',
                F.upper(F.trim(F.col('customer_state'))))\
    .withColumn('ingestion_timestamp',F.current_timestamp())\
    .withColumn('source_system',F.lit('olist_customers_dataset'))

In [0]:
display(df_silver_customer.limit(5))

In [0]:
null_count_customer=df_silver_customer.select([
   F.count(F.when(F.col(colmn).isNull(),colmn)).alias(colmn)
      for colmn in df_silver_customer.columns
])

display(null_count_customer)

In [0]:
state_wise_count=df_silver_customer.groupBy('customer_state')\
.agg(F.count('*').alias('Total_count'))\
.orderBy('Total_count',ascending=False)
display(state_wise_count)

In [0]:
display(state_wise_count.filter((F.col('Total_count')>10000)))

In [0]:
display(state_wise_count.filter((F.col('Total_count')>1000) & (F.col('Total_count')<10000)))

In [0]:
display(state_wise_count.filter(F.col('Total_count')<1000))

In [0]:
display(state_wise_count.agg(F.sum('Total_count').alias('Total')))

In [0]:
city_wise=df_silver_customer.groupBy('customer_city')\
    .agg(F.count('*').alias('Total_city'))\
        .orderBy('Total_city',descending=True)
display(city_wise)

In [0]:
display(city_wise.filter(F.col('Total_city')>100))

In [0]:
df_silver_customer.write\
    .format('delta')\
    .mode('overwrite')\
    .option('overwriteSchema',True)\
    .save(f'abfss://silver@stretailanalytics.dfs.core.windows.net/SILVER_CUSTOMER/')

In [0]:
df_silver_customer_check=spark.read\
    .format('delta')\
        .load(f'abfss://silver@stretailanalytics.dfs.core.windows.net/SILVER_CUSTOMER/')
display(df_silver_customer_check.limit(10))

Payment Silver 

In [0]:
payment_schema=StructType([
    StructField('order_id',StringType(),False),
    StructField('payment_sequential',IntegerType(),True),
    StructField('payment_type',StringType(),True),
    StructField('payment_installments',IntegerType(),True),
    StructField('payment_value',DoubleType(),True)
])

In [0]:
df_raw_payment=spark.read\
    .schema(payment_schema)\
    .option('header',True)\
    .csv(f'abfss://bronze@{storage_account_name}.dfs.core.windows.net/staged/payments/')

print("Silver dataframe created")

In [0]:
display(df_raw_payment.limit(10))

In [0]:
null_count=df_raw_payment.select([
   F.count(F.when(F.col(c).isNull(),c)).alias(c) for c in df_raw_payment.columns
])
display(null_count)

In [0]:
df_silver_payment=df_raw_payment.dropna(subset=['order_id'])\
    .dropDuplicates(['order_id'])\
    .filter(F.col('payment_value')>0)\
    .withColumn('payment_type',
                F.initcap(F.trim(F.col('payment_type'))))\
    .withColumn('payment_value', F.round(F.col('payment_value')))\
    .withColumn('ingestion_timestamp', F.current_timestamp())\
    .withColumn('source_data', F.lit('olist_order_payments'))

print('Transformation Done')

display(df_silver_payment.limit(15))

In [0]:
count_=df_silver_payment.groupBy('payment_type')\
    .agg(F.count('*').alias('Payment_type_count'))
display(count_)

In [0]:
from pyspark.sql.window import Window

window_spec=Window.orderBy(F.col('Payment_type_count').desc())
ranking=count_.withColumn('Rank',
                          F.rank().over(window_spec))
display(ranking)

In [0]:
df_silver_payment.write\
    .format('delta')\
    .mode('overwrite')\
    .option('overwriteSchema',True)\
    .partitionBy('payment_type')\
    .save(f'abfss://silver@stretailanalytics.dfs.core.windows.net/SILVER_PAYMENT/')

print('Silver_payment data saved successfully!!')

In [0]:
df_payment_check=spark.read\
    .format('delta')\
    .option('header',True)\
    .load(f'abfss://silver@stretailanalytics.dfs.core.windows.net/SILVER_PAYMENT/payment_type=Boleto/')
display(df_payment_check.limit(10))

Order Item Silver

In [0]:
item_schema=StructType([
    StructField('order_id',StringType(),False),
    StructField('order_item_id',IntegerType(),True),
    StructField('product_id',StringType(),False),
    StructField('seller_id',StringType(),False),
    StructField('shipping_limit_date',TimestampType(),True),
    StructField('price',DoubleType(),True),
    StructField('freight_value',DoubleType(),True)
])

In [0]:
df_raw_item=spark.read\
    .schema(item_schema)\
    .option('header',True)\
    .option('timestampFormat','yyyy-MM-dd HH:mm:ss')\
    .csv(f'abfss://bronze@{storage_account_name}.dfs.core.windows.net/staged/order_items/')

display(df_raw_item.limit(10))

In [0]:
null_values_item=df_raw_item.select([F.count(F.when(F.col(c).isNull(),c)).alias(c) for c in df_raw_item.columns])
display(null_values_item)

In [0]:
df_item_silver=df_raw_item.dropna(subset=['order_id','product_id','seller_id'])\
    .withColumn('shipping_year', F.year('shipping_limit_date'))\
    .withColumn('shipping_month', F.month('shipping_limit_date'))\
    .withColumn('total_price', F.round((F.col('price')+F.col('freight_value')),2))\
    .withColumn('ingestion_time', F.current_timestamp())\
    .withColumn('source_data',F.lit('olist_order_items_dataset'))

print('Transformation Done')

In [0]:
display(df_item_silver.limit(10))

In [0]:
df_item_silver.write\
    .format('delta')\
    .mode('overwrite')\
    .option('overwriteSchema',True)\
    .partitionBy('shipping_year')\
    .save(f'abfss://silver@{storage_account_name}.dfs.core.windows.net/SILVER_ITEMS')

print('Order_item written in Silver container')

In [0]:
display(dbutils.fs.ls(f'abfss://silver@{storage_account_name}.dfs.core.windows.net/'))